# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

In [8]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import os
from matplotlib import pyplot as plt

import importlib
import ALC
import MODULO_ALC

from MODULO_ALC import rq, svd_reducida, calculaQR, multiplicacionMatricial, inversa, traspuesta, QR_con_HH, calculaCholesky, norma, productoExterno
from ALC import determinante3x3, estimarH, armarA, calcularMatrizK2D

## GRUPO: A=TR

* Franco Nahuel Chazarreta Villagra
* Julián Nakasone
* Gabriel Bravo
* Pedro Soldatich

### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

## Ejercicio 1 (a): DLT en 3 dimensiones

### Construcción de la matriz Q (Introduccion)

Podemos observar que en `detected_corners.txt` cada fila corresponde a una imagen. Como cada punto detectado tiene dos coordenadas, $(u_i,v_i)$, y cada fila contiene 24 columnas, significa que tenemos 12 puntos detectados por imagen. Por lo tanto, podemos representar cada fila de la siguiente manera:

$$
[u_0,v_0,u_1,v_1,\ldots,u_{11},v_{11}]
$$

**Ecuaciones de una correspondencia.** Para cada correspondencia 3D-2D ($i=0,\ldots,11$) tenemos un punto del objeto, en coordenadas homogéneas:

$$
P_i=(U_i,V_i,W_i,1)^\top
$$

y su píxel correspondiente $(u_i,v_i)$. La ecuación (3) del enunciado dice:

$$
\lambda_i
\begin{bmatrix}
u_i\\
v_i\\
1
\end{bmatrix}
=MP_i,
\qquad
M=
\begin{bmatrix}
m_1^\top\\
m_2^\top\\
m_3^\top
\end{bmatrix}
\in\mathbb R^{3\times4}.
$$

Desarrollando el producto fila por fila (cada $m_j^\top P_i$ es un número), obtenemos:

$$
m_1^\top P_i=\lambda_i u_i,
\qquad
m_2^\top P_i=\lambda_i v_i,
\qquad
m_3^\top P_i=\lambda_i.
$$

$\lambda_i$ es el factor de escala de cada punto (es proporcional a su profundidad), por lo que es una incógnita distinta para cada correspondencia. Para no agregar una incógnita por punto, la eliminamos reemplazando la tercera ecuación en las otras dos:

$$
(m_1^\top-u_im_3^\top)P_i=0,
\qquad
(m_2^\top-v_im_3^\top)P_i=0.
$$

Estas ecuaciones son **lineales** en las 12 entradas de $M$, porque $u_i$, $v_i$ y $P_i$ son datos conocidos.

Apilamos las filas de $M$, escritas como columnas, en el vector:

$$
m=
\begin{bmatrix}
m_1\\
m_2\\
m_3
\end{bmatrix}
\in\mathbb R^{12}.
$$

Como un número es igual a su transpuesto, $m_j^\top P_i=P_i^\top m_j$, y entonces

$$
(m_1^\top-u_im_3^\top)P_i=P_i^\top m_1+0^\top m_2-u_iP_i^\top m_3=
\begin{bmatrix}P_i^\top&0^\top&-u_iP_i^\top\end{bmatrix}m .
$$

Haciendo lo mismo con la ecuación de $v_i$, cada punto aporta dos filas de $Q$ (ecuaciones 5 y 6 del enunciado):

$$
\begin{bmatrix}
U_i&V_i&W_i&1&0&0&0&0&
-u_iU_i&-u_iV_i&-u_iW_i&-u_i\\
0&0&0&0&U_i&V_i&W_i&1&
-v_iU_i&-v_iV_i&-v_iW_i&-v_i
\end{bmatrix}
m=
\begin{bmatrix}
0\\
0
\end{bmatrix}.
$$

Entonces, como tenemos 12 puntos por imagen y cada uno aporta dos ecuaciones, obtenemos:

$$
Q\in\mathbb R^{24\times12}.
$$

En otras palabras, tendremos 24 ecuaciones y 12 incógnitas por cada imagen.

**¿La matriz de proyección $M_j$ es igual para todas las imágenes?**

No. Cada imagen tiene una posición y orientación relativas diferentes entre la cámara y el objeto. Por lo tanto, sus parámetros extrínsecos, es decir, la matriz de rotación $R_j$ y el vector de traslación $t_j$, también son diferentes.

Sin embargo, como estamos utilizando la misma cámara, sus parámetros intrínsecos $K$ permanecen constantes (suponiendo que no se modifica la configuración de la cámara entre capturas).

Por lo tanto, para cada imagen $j$ tenemos una matriz de proyección diferente:

$$
\boxed{M_j=K[R_j\mid t_j]}
$$

En consecuencia, debemos construir una matriz $Q_j$ independiente para cada imagen, ya que cada una permite estimar su propia matriz de proyección $M_j$.

---

### ¿Qué rango debe tener Q para obtener una solución no nula única, salvo escala?

Queremos determinar el rango que debe tener la matriz $Q$ para que el sistema homogéneo:

$$
Qm=0
$$

tenga una solución no nula única, salvo un factor de escala. Sabemos que:

$$
Q\in\mathbb R^{2n\times12}
$$

(con $n$ la cantidad de puntos) y que:

$$
m\in\mathbb R^{12}.
$$

Recordemos las definiciones:

$$
\ker(Q)=\{x\in\mathbb R^{12}:Qx=0\},
$$

$$
\operatorname{im}(Q)=
\{b\in\mathbb R^{2n}:b=Qx,\text{ para algún }x\in\mathbb R^{12}\},
$$

$$
\operatorname{rango}(Q)=\dim(\operatorname{im}(Q)).
$$

Las soluciones del sistema $Qm=0$ son, por definición, los vectores de $\ker(Q)$.

Para que exista una única solución no nula salvo escala, todas las soluciones deben ser múltiplos de un mismo vector, es decir:

$$
\dim(\ker(Q))=1.
$$

**Teorema de la dimensión.** Dada una matriz $A\in\mathbb K^{p\times q}$ (usamos $p\times q$ en lugar de $m\times n$ para no pisar las letras $m$, vector de incógnitas, y $n$, cantidad de puntos), se cumple que:

$$
q=\dim(\ker(A))+\dim(\operatorname{im}(A)),
$$

donde $q$ es la cantidad de columnas de $A$.

Nuestra matriz $Q$ tiene 12 columnas, entonces:

$$
12=\dim(\ker(Q))+\dim(\operatorname{im}(Q))
$$

$$
12=\dim(\ker(Q))+\operatorname{rango}(Q).
$$

Como necesitamos que $\dim(\ker(Q))=1$:

$$
12=1+\operatorname{rango}(Q)
$$

Por lo tanto:

$$
\boxed{\operatorname{rango}(Q)=11}
$$

**¿Qué pasa si el rango no es 11?**

- Si $\operatorname{rango}(Q)<11$, entonces $\dim(\ker(Q))\geq2$. El conjunto de soluciones es un subespacio de dimensión al menos 2: existen dos soluciones linealmente independientes y, por lo tanto, infinitas soluciones no proporcionales entre sí. Hay más de una variable libre y no se puede determinar $M$ de manera única salvo escala.

- Si $\operatorname{rango}(Q)=12$, entonces $\ker(Q)=\{0\}$. La única solución es $m=0$, que no representa ninguna cámara.

**Observación (datos exactos vs. datos reales).**

Con datos exactos, el vector $m$ verdadero (no nulo) pertenece a $\ker(Q)$, por lo que:

$$
\operatorname{rango}(Q)\leq11.
$$

Lo que necesitamos asegurar es que sea exactamente 11.

Con datos reales (errores al marcar los píxeles, ruido, errores al medir las coordenadas del objeto, etc.), $Q$ suele tener rango 12 y la única solución exacta de $Qm=0$ es $m=0$.

Por eso, en el inciso (b), la igualdad exacta se reemplaza por minimizar:

$$
\boxed{
\min_{\|m\|_2=1}\|Qm\|_2^2
}
$$

como se plantea en la ecuación 7 del enunciado.

---

### ¿Cuántos puntos se necesitan como mínimo?

Sabemos que:

$$
Q\in\mathbb R^{2n\times12}
$$

y que necesitamos:

$$
\operatorname{rango}(Q)=11.
$$

La imagen de una matriz está generada por sus columnas, porque:

$$
Qx=x_1c_1+x_2c_2+\dots+x_{12}c_{12},
$$

donde $c_1,\dots,c_{12}$ son las columnas de $Q$: todo vector de $\operatorname{im}(Q)$ es una combinación lineal de ellas.

Además, $\operatorname{im}(Q)$ es un subespacio de $\mathbb R^{2n}$.

De ahí salen dos cotas para el rango:

$$
\operatorname{rango}(Q)
=\dim(\operatorname{im}(Q))
\leq 2n
\quad
(\text{subespacio de }\mathbb R^{2n}),
$$

$$
\operatorname{rango}(Q)\leq12
\quad
(\text{generado por 12 columnas}).
$$

Para que el rango pueda alcanzar el valor 11, necesariamente debe cumplirse:

$$
2n\geq11
\quad\Longrightarrow\quad
n\geq\frac{11}{2}.
$$

Como $n$ es una cantidad de puntos, debe ser entero. Por lo tanto:

$$
\boxed{n_{\min}=6\text{ puntos}}
$$

Por ejemplo, con $n=5$ tendríamos 10 ecuaciones, entonces:

$$
\operatorname{rango}(Q)\leq10
$$

y, por el teorema de la dimensión:

$$
\dim(\ker(Q))\geq2.
$$

Por lo tanto, no habría unicidad.

Es importante aclarar que 6 puntos son el mínimo necesario por cantidad de ecuaciones, pero no garantizan por sí solos una solución única. Los puntos también deben cumplir ciertas condiciones geométricas.

---

### ¿Qué condición geométrica deben cumplir los puntos?

Para que $Q$ tenga rango 11 no alcanza con tener $2n\geq11$ filas: la imagen de $Q$, generada por sus 12 columnas, tiene que tener dimensión 11. Si los puntos están mal ubicados, hay columnas que no aportan una dirección nueva (por ejemplo, columnas nulas) y el rango baja.

**Condición necesaria: los puntos no pueden ser todos coplanares.**

Veamos qué pasa si lo están. Existe un cambio de coordenadas del objeto $P_i'=TP_i$, con $T\in\mathbb R^{4\times4}$ invertible, tal que ese plano pasa a ser $W=0$. Los píxeles no cambian, y las filas de la nueva matriz $Q'$ son las de $Q$ multiplicadas a derecha por la matriz $\operatorname{diag}(T^\top,T^\top,T^\top)\in\mathbb R^{12\times12}$, que es invertible. Por lo tanto $\operatorname{rango}(Q')=\operatorname{rango}(Q)$, y alcanza con analizar el caso $W=0$.

Entonces:

$$
P_i=(U_i,V_i,0,1)^\top
$$

para todo $i$, y las dos filas de cada punto quedan:

$$
\begin{bmatrix}
U_i&V_i&\mathbf{0}&1&0&0&\mathbf{0}&0&
-u_iU_i&-u_iV_i&\mathbf{0}&-u_i\\
0&0&\mathbf{0}&0&U_i&V_i&\mathbf{0}&1&
-v_iU_i&-v_iV_i&\mathbf{0}&-v_i
\end{bmatrix}.
$$

Las columnas 3, 7 y 11 (las que multiplican a $W_i$) son **idénticamente nulas**, para todos los puntos.

Entonces, $\operatorname{im}(Q)$ está generada por las otras 9 columnas, y:

$$
\operatorname{rango}(Q)
=\dim(\operatorname{im}(Q))
\leq9<11.
$$

Por el teorema de la dimensión:

$$
\dim(\ker(Q))=12-\operatorname{rango}(Q)\geq3.
$$

En otras palabras, las incógnitas $m_{13}$, $m_{23}$ y $m_{33}$ ni siquiera aparecen en las ecuaciones: sus valores no pueden determinarse a partir de estas correspondencias.

**Interpretación:** si todos los puntos están en $W=0$, los datos no dicen nada sobre cómo la cámara trata la coordenada $W$, que es justamente la que hace falta para describir un objeto en 3D. Las 9 columnas restantes forman el sistema de la homografía, que se resuelve en el Ejercicio 3.

**Conclusión.** Para poder obtener:

$$
\operatorname{rango}(Q)=11
$$

necesitamos, como mínimo:

$$
\boxed{
n\geq6
\qquad\text{y}\qquad
\text{los puntos no son todos coplanares}.
}
$$

Estas son condiciones necesarias, pero no suficientes por sí solas: también hay que evitar otras configuraciones geométricas degeneradas. Por eso, en la práctica, los puntos deben estar bien distribuidos en el volumen del objeto y se utilizan más de 6 correspondencias.

**Verificación en nuestros datos.** En nuestro caso las 12 esquinas están repartidas en 3 hojas del mueble. Cada hoja es plana, pero eso no alcanza para afirmar que el conjunto completo no es coplanar, así que lo comprobamos. Sea $\mathcal P\in\mathbb R^{12\times4}$ la matriz cuyas filas son los $P_i^\top$ (los puntos de `p_W_corners.txt` en coordenadas homogéneas). Los puntos son coplanares si y sólo si existe $\pi\neq0$ con $\mathcal P\pi=0$, es decir, si y sólo si $\operatorname{rango}(\mathcal P)<4$. Entonces basta verificar que $\operatorname{rango}(\mathcal P)=4$, o sea que su cuarto valor singular no sea del orden del error de máquina.

Con los datos reales, por el ruido, el rango numérico de $Q$ es 12. Lo que esperamos observar es que $\sigma_{12}$ sea mucho menor que $\sigma_{11}$ (rango "efectivo" 11), y por eso en el inciso (b) se resuelve la minimización de la ecuación 7.

--

## Ejercicio 1 (b): Encontrar el vector $m$ que resuelve la minimización

Partimos de la ecuación 8 del enunciado:

$$
\boxed{
\min_{\|m\|_2=1}m^\top Q^\top Qm
}
$$

Primero, recordemos que:

$$
m^\top Q^\top Qm=(Qm)^\top(Qm)=\|Qm\|_2^2
$$

Por lo tanto, podemos expresar el problema como:

$$
\boxed{
\min_{\|m\|_2=1}\|Qm\|_2^2
}
$$

La restricción $\|m\|_2=1$ evita que la solución trivial $m=0$ sea el mínimo.


### Resolución mediante SVD

**Definición 7.1 (SVD).** Dada $A\in\mathbb R^{p\times q}$ (usamos $p\times q$ en lugar de $m\times n$ para no pisar la letra $m$ del vector de incógnitas), la descomposición en valores singulares es una factorización

$$
A=U\Sigma V^\top,
$$

con $U\in\mathbb R^{p\times p}$ y $V\in\mathbb R^{q\times q}$ matrices ortogonales (Definición 3.7; como trabajamos en $\mathbb R$, $V^*=V^\top$) y $\Sigma\in\mathbb R^{p\times q}$. Si $p\geq q$, la matriz $\Sigma$ tiene la forma

$$
\Sigma=\begin{pmatrix}D\\ 0_{(p-q)\times q}\end{pmatrix},
\qquad
D=\operatorname{diag}(\sigma_1,\dots,\sigma_q),
\qquad
\sigma_1\geq\sigma_2\geq\cdots\geq\sigma_q\geq0 .
$$

Los $\sigma_i$ son los valores singulares de $A$ y las columnas de $V$ son sus vectores singulares derechos.

Como queremos minimizar la norma de $Qm$, aplicamos esta definición a $A=Q$, que tiene más filas que columnas ($24\geq12$):

$$
Q=U\Sigma V^\top
$$

Donde, para nuestro caso, tenemos:

$$
Q\in\mathbb R^{24\times12},
\qquad
U\in\mathbb R^{24\times24},
\qquad
\Sigma=\begin{pmatrix}D\\ 0_{12\times12}\end{pmatrix}\in\mathbb R^{24\times12},
\qquad
V\in\mathbb R^{12\times12}.
$$

Reemplazamos la descomposición SVD en nuestra función objetivo. Como $U$ es ortogonal, se cumple que $U^\top U=I$ (Definición 3.7). Entonces:

$$
\begin{aligned}
\|Qm\|_2^2
&=(U\Sigma V^\top m)^\top(U\Sigma V^\top m)\\
&=m^\top V\Sigma^\top U^\top U\Sigma V^\top m\\
&=m^\top V\Sigma^\top\Sigma V^\top m.
\end{aligned}
$$

Además, por la forma de $\Sigma$, resulta $\Sigma^\top\Sigma=D^\top D=D^2$ (el bloque de ceros no aporta nada).

Ahora, definimos:

$$
z=V^\top m
$$

Como $V$ es ortogonal, también lo es $V^\top$ (Observación 3.3), y se cumple $VV^\top=I$. Entonces:

$$
\|z\|_2^2=m^\top VV^\top m=\|m\|_2^2=1.
$$

Por lo tanto, nuestra función objetivo queda:

$$
\begin{aligned}
\|Qm\|_2^2
&=z^\top D^2 z\\
&=\sigma_1^2z_1^2+\sigma_2^2z_2^2+\cdots+\sigma_{12}^2z_{12}^2,
\end{aligned}
\qquad\text{con}\qquad
z_1^2+z_2^2+\cdots+z_{12}^2=1.
$$

Como los valores singulares están ordenados de mayor a menor, $\sigma_i\geq\sigma_{12}$ para todo $i$, y entonces:

$$
\sigma_1^2z_1^2+\cdots+\sigma_{12}^2z_{12}^2
\;\geq\;
\sigma_{12}^2\,(z_1^2+\cdots+z_{12}^2)
=\sigma_{12}^2 .
$$

Es decir, $\|Qm\|_2^2\geq\sigma_{12}^2$, y la cota se alcanza cuando toda la contribución corresponde al menor valor singular:

$$
z=
\begin{bmatrix}
0\\
0\\
\vdots\\
0\\
\pm1
\end{bmatrix}.
$$

Como habíamos definido $z=V^\top m$, multiplicando por $V$ obtenemos $m=Vz$ (porque $VV^\top=I$). Por lo tanto:

$$
\boxed{m=\pm v_{12}}
$$

Donde $v_{12}$ es la última columna de $V$, correspondiente al menor valor singular de $Q$. Si $\sigma_{11}>\sigma_{12}$, estos son los únicos mínimos: la solución es única salvo el signo.

### Recuperación de la matriz de proyección M

Una vez obtenido el vector $m$, podemos reconstruir la matriz de proyección $M$.

Recordemos que habíamos definido $m=(m_1;m_2;m_3)$, es decir, las filas de $M$ una detrás de otra:

$$
m=
\begin{bmatrix}
m_{11}&m_{12}&m_{13}&m_{14}&m_{21}&m_{22}&m_{23}&m_{24}&m_{31}&m_{32}&m_{33}&m_{34}
\end{bmatrix}^\top.
$$

Por lo tanto, reorganizando sus componentes obtenemos:

$$
\boxed{
M=
\begin{bmatrix}
m_{11}&m_{12}&m_{13}&m_{14}\\
m_{21}&m_{22}&m_{23}&m_{24}\\
m_{31}&m_{32}&m_{33}&m_{34}
\end{bmatrix}
}
$$

### Conclusión

La solución de la minimización de la ecuación 8 se obtiene directamente mediante la descomposición SVD de $Q$.

Por lo tanto, debemos calcular:

$$
Q=U\Sigma V^\top
$$

Y seleccionar la última columna de $V$, suponiendo que los valores singulares están ordenados de mayor a menor:

$$
\boxed{m=\pm V[:,11]}
$$

(índices contando desde 0). Finalmente, reorganizamos las 12 componentes de $m$ para obtener la matriz de proyección $M$ de dimensión $3\times4$.

La restricción $\|m\|_2=1$ fija la escala de $m$, así que la solución queda determinada salvo el signo. En cuanto a $M$, queda definida a menos de signo y escala, como corresponde al modelo de proyección perspectiva, y ambos se corrigen al descomponerla en $K$, $R$ y $t$ (Ejercicio 2c).

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

In [ ]:
imgPts_todas = np.loadtxt('calib/3d/detected_corners.txt')
objPts = np.loadtxt('calib/3d/p_W_corners.txt', delimiter=',')

imgPts = imgPts_todas[0].reshape(12, 2)
Q = ALC.obtenerQ3d(imgPts, objPts)
M = ALC.resolverDLT3D(Q)
R, t, K = ALC.devolverMatricesCalibracion3d(M)

print("K:")
print(K)

print("R:")
print(R)

print("t:")
print(t)

print("Y comprobemos que el determinante de R es 1")

print("Determinante de R:")
print(determinante3x3(R))

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



In [ ]:
print("parte b")

imgPts_todas = np.loadtxt('calib/3d/detected_corners.txt')
objPts = np.loadtxt('calib/3d/p_W_corners.txt', delimiter=',')

imgPts = imgPts_todas[0].reshape(12, 2)
Q = ALC.obtenerQ3d(imgPts, objPts)
M = ALC.resolverDLT3D(Q)
R, t, K = ALC.devolverMatricesCalibracion3d(M, True)

print("K:")
print(K)

print("R:")
print(R)

print("t:")
print(t)

print("Y comprobemos que el determinante de R es 1")

print("Determinante de R:")
print(determinante3x3(R))

In [ ]:
print("parte c")

imgPts_todas = np.loadtxt('calib/3d/detected_corners.txt')
objPts = np.loadtxt('calib/3d/p_W_corners.txt', delimiter=',')

imgPts = imgPts_todas[0].reshape(12, 2)
Q = ALC.obtenerQ3d(imgPts, objPts)
M = ALC.resolverDLT3D(Q)
R, t, K = ALC.devolverMatricesCalibracion3d(M, False, True)

print("K:")
print(K)

print("R:")
print(R)

print("t:")
print(t)

print("Y comprobemos que el determinante de R es 1")

print("Determinante de R:")
print(determinante3x3(R))

In [ ]:
print("ambas restricciones son verdaderas")
imgPts_todas = np.loadtxt('calib/3d/detected_corners.txt')
objPts = np.loadtxt('calib/3d/p_W_corners.txt', delimiter=',')

imgPts = imgPts_todas[0].reshape(12, 2)
Q = ALC.obtenerQ3d(imgPts, objPts)
M = ALC.resolverDLT3D(Q)
R, t, K = ALC.devolverMatricesCalibracion3d(M, True, True)

print("K:")
print(K)

print("R:")
print(R)

print("t:")
print(t)

print("Y comprobemos que el determinante de R es 1")

print("Determinante de R:")
print(determinante3x3(R))

Conclusiones

Se verificó la implementación con las vistas 1 y 50. En ambas:

La matriz R resultó ortogonal y con determinante cercano a 1.
La matriz K resultó triangular superior, con diagonal positiva y K[2,2] = 1.
La reconstrucción de M, salvo escala, presentó errores del orden de 10⁻¹⁶, compatibles con el redondeo numérico.
Los doce puntos quedaron delante de la cámara.

| Vista | RMS sin restricciones | RMS con focales iguales y sesgo nulo |
|---    |---:                   |---:                                  |
| 1     | 0.407 px              | 0.571 px                             |
| 50    | 0.495 px              | 0.727 px                             |

Los errores de reproyección fueron menores a un píxel en las dos vistas. Al imponer las restricciones sobre K, el RMS aumentó moderadamente, pero se mantuvo por debajo de un píxel.

Las restricciones se aplicaron después de la descomposición: se promediaron las focales y se anuló el sesgo. Esto impone las condiciones solicitadas, aunque no constituye una optimización del error de reproyección bajo restricciones.

Estos resultados respaldan el funcionamiento de la implementación para las dos vistas evaluadas.

### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$
## 1. Introducción

Ahora trabajamos con puntos coplanares, es decir, puntos que pertenecen a un mismo plano.

Disponemos de un tablero de ajedrez de $12\times 8$ cuadrados, por lo que tiene $11\times 7$ esquinas internas. Cada cuadrado mide $30\,\text{mm}\times 30\,\text{mm}$, y del cual tomamos como sistema de referencia del tablero un origen en una de sus esquinas internas, con los ejes $U$ y $V$ a lo largo de sus lados y $W=0$ (ya que todos los puntos pertenecen a un mismo plano).

Por lo tanto, las coordenadas de las esquinas internas son:

$$
U=30i,\qquad V=30j,\qquad W=0
$$

donde:

$$
i=0,\ldots,10,\qquad j=0,\ldots,6
$$

En total, tenemos:

$$
n=11\cdot 7=77
$$

correspondencias entre puntos del tablero y sus posiciones en la imagen.

Queremos estimar una homografía $H_j$ para cada una de las 20 fotografías del tablero, utilizando el método DLT.

---

## 2. Planteo del sistema

Como $W=0$, la tercera columna de $R$ queda multiplicada por cero, y la ecuación de proyección se reduce a las ecuaciones (9) y (10) del enunciado:

$$
\lambda\,m=HP,\qquad H=K\,[r_1\ \ r_2\ \ t]\in\mathbb{R}^{3\times3},
$$

con $P=(U,V,1)^\top$ y $m=(u,v,1)^\top$. Como $\lambda$ absorbe cualquier factor de escala, $H$ está definida salvo escala (9 entradas, 8 grados de libertad).

Igual que en el Ejercicio 1, se abre la ecuación fila por fila y se elimina $\lambda_i$ reemplazando la tercera igualdad en las otras dos:

$$
(h_1^\top-u_ih_3^\top)P_i=0,\qquad (h_2^\top-v_ih_3^\top)P_i=0,
$$

donde $h_j^\top$ es la fila $j$ de $H$. Son ecuaciones lineales en $h=(h_1;h_2;h_3)\in\mathbb{R}^9$ (las filas de $H$ una detrás de otra). Cada punto aporta dos filas:

$$
Q_i=
\begin{bmatrix}
U_i&V_i&1&0&0&0&-u_iU_i&-u_iV_i&-u_i\\
0&0&0&U_i&V_i&1&-v_iU_i&-v_iV_i&-v_i
\end{bmatrix}.
$$

Apilando las $n$ correspondencias (ecuación 11 del enunciado):

$$
\boxed{Qh=0},\qquad \boxed{Q\in\mathbb{R}^{2n\times9}},\qquad Q\in\mathbb{R}^{154\times9}\ \text{en nuestro caso}.
$$

---

## 3. Rango de $Q$ y cantidad de puntos

Queremos que la solución sea única salvo escala, es decir, $\dim(\ker(Q))=1$. Por el teorema de la dimensión (Teorema 1.1), como $Q$ tiene 9 columnas:

$$
9=\dim(\ker(Q))+\operatorname{rango}(Q)\quad\Longrightarrow\quad \boxed{\operatorname{rango}(Q)=8}.
$$

Como $\operatorname{im}(Q)\subseteq\mathbb{R}^{2n}$, se cumple $\operatorname{rango}(Q)\leq2n$, y para que el rango sea 8 hace falta $2n\geq8$:

$$
\boxed{n\geq4}.
$$

---
b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

In [ ]:
import os
import numpy as np
import ALC

np.set_printoptions(precision=8, suppress=True)

carpeta = os.path.join("calib", "coplanar")

# Mismo orden que los clics: sup. izq., sup. der., inf. der., inf. izq.
objPts = np.array([
    [0,   0],
    [300, 0],
    [300, 180],
    [0,   180]
])

# Estimamos una homografía por imagen.
Hlst = []

for numero in range(1, 21):
    imgPts = np.loadtxt(os.path.join(carpeta, f"Im_R_{numero}_4puntos.txt")
    )

    H = ALC.estimarH(imgPts, objPts)
    Hlst.append(H)

# a) Una matriz intrínseca común a las veinte vistas.

A = armarA(Hlst)
K = calcularMatrizK2D(A)

print("a) Matriz K:")
print(K)

# b) Poses de las imágenes solicitadas.
for numero in [1, 10, 20]:
    R, t = ALC.obtenerPose2D(Hlst[numero - 1], K)

    print(f"\nb) Pose de Im_R_{numero}.png")
    print("R, de objeto a cámara:")
    print(R)
    print("t [mm]:")
    print(t)

print(
    "\nRango requerido de Q: 8, para determinar "
    "una homografía única salvo escala."
)

Conclusiones del ejercicio 3
Se estimó una matriz intrínseca K común a las veinte vistas utilizando cuatro esquinas internas extremas del tablero por imagen. Las coordenadas reales se expresaron en milímetros, con separaciones de 30 mm entre esquinas consecutivas.

Las focales estimadas fueron αu = 1080.51 px y αv = 1103.41 px, con una diferencia relativa aproximada del 2.1 %. El punto principal resultó (631.14, 378.52) px y el sesgo fue −19.22 px. La matriz K obtenida es triangular superior, tiene diagonal positiva y está normalizada con K33 = 1.

Se recuperaron las poses de las imágenes 1, 10 y 20 utilizando la convención P_cámara = R P_objeto + t. La rotación de la primera vista es cercana a la identidad, mientras que las otras dos presentan
mayores inclinaciones del tablero respecto de la cámara.

| Imagen | tx [mm] | ty [mm] | tz [mm] |
|---     |---:     |---:     |---:     |
| 1      | −194.41 | −198.54 | 797.57  |
| 10     | −180.14 | −206.70 | 782.63  |
| 20     | −163.18 | −199.79 | 723.66  |

La componente tz representa la profundidad del origen del tablero en el sistema de la cámara. Esta disminuye entre las vistas seleccionadas, pasando de aproximadamente 798 mm a 724 mm; no debe confundirse con la distancia euclídea entre la cámara y dicho origen.

Para determinar una homografía única salvo escala, la matriz Q debe tener rango 8, ya que contiene nueve incógnitas homogéneas.

El uso de cuatro correspondencias por imagen permite estimar las homografías, pero hace que la calibración sea sensible a los errores del marcado manual. Sin una evaluación del error de reproyección, estos resultados permiten describir la calibración estimada, pero no cuantificar su precisión.

### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

In [ ]:
# a) Distancia de Enzo a la línea del arco

img = cv2.imread('futbol/test-gol-enzo.png')
img_rgb = img[:, :, ::-1]

# Mostramos la imagen con grilla para leer a ojo las coordenadas (u, v)
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.xlabel('u (píxeles horizontal)')
plt.ylabel('v (píxeles vertical)')
plt.title('Identificación de píxeles (u, v)')
plt.show()

In [ ]:
# Puntos de referencia en la imagen (u, v), leídos del gráfico anterior
imgPts = np.array([
    [160, 400],  # 1. Esquina frontal izq del área grande
    [980, 400],  # 2. Esquina frontal der del área grande
    [280,  98],  # 3. Línea de meta / área grande (izq)
    [745,  98],  # 4. Línea de meta / área grande (der)
    [362, 175],  # 5. Esquina frontal izq del área chica
    [628, 175],  # 6. Esquina frontal der del área chica
    [478, 255],  # 7. Punto penal
], dtype=np.float32)

# Los mismos puntos en coordenadas de la cancha (X, Y) en metros,
# con el origen en la esquina del arco y X medido desde la línea de meta.
# Mismo orden que imgPts (el punto i de uno corresponde al punto i del otro)
objPts = np.array([
    [16.5, 13.84],  # 1. Esquina frontal izq del área grande
    [16.5, 54.16],  # 2. Esquina frontal der del área grande
    [ 0.0, 13.84],  # 3. Línea de meta / área grande (izq)
    [ 0.0, 54.16],  # 4. Línea de meta / área grande (der)
    [ 5.5, 24.84],  # 5. Esquina frontal izq del área chica
    [ 5.5, 43.16],  # 6. Esquina frontal der del área chica
    [11.0, 34.00],  # 7. Punto penal
], dtype=np.float32)

# DLT con normalización de Hartley (estimarH arma Q con obtenerQ2d y resuelve con SVD)
H = estimarH(imgPts.astype(np.float64), objPts.astype(np.float64))

# ------------------------------------------------------------
# Posición de los pies de Enzo en la cancha
# ------------------------------------------------------------
u = 810   # píxel (u, v) de los pies de Enzo en la imagen
v = 520

# Llevamos el píxel a la cancha con la inversa de H
pt_imagen = np.array([[u], [v], [1]])

pt_real = multiplicacionMatricial(inversa(H), pt_imagen)

# Pasamos de coordenadas homogéneas a (X, Y) dividiendo por la tercera componente
xReal = float(pt_real[0, 0] / pt_real[2, 0])
yReal = float(pt_real[1, 0] / pt_real[2, 0])

# La línea de meta es X = 0, así que la distancia es |X|
dist_linea_meta = abs(xReal)
print(f"Coordenadas reales (X, Y): ({xReal:.2f}, {yReal:.2f})")
print(f"Distancia a la línea de meta: {dist_linea_meta:.2f} metros")

Coordenadas reales (X, Y): (19.16, 45.98)
Distancia a la línea de meta: 19.16 metros


El campo de juego es un plano, así que la relación entre sus puntos
$(U, V, 1)^T$ y los píxeles $(u, v, 1)^T$ de la foto es una homografía $H$:

$$\lambda \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = H \begin{bmatrix} U \\ V \\ 1 \end{bmatrix}$$

Para este ejercicio no necesitamos $K$ ni descomponer $H$ en $R$ y $t$: alcanza con $H$ y su inversa.

**Procedimiento.**
1. Identificamos 7 puntos de referencia en la imagen (esquinas del área grande
   y del área chica, intersecciones con la línea de meta y punto penal) y los
   emparejamos con sus coordenadas en metros en la cancha.
2. Estimamos $H$ con DLT (sistema $Qh = 0$, mínimos cuadrados homogéneos vía SVD),
   usando normalización de Hartley para mejorar el condicionamiento.
3. Verificamos $H$ reproyectando los puntos de la cancha a la imagen y calculando el RMS.
4. Llevamos los pies de Enzo, $(u, v)$, a la cancha con $H^{-1}$ y dividimos por la
   tercera componente para volver de coordenadas homogéneas:
   $(X, Y) = (x'/z', \, y'/z')$.
5. Como la línea del arco es $X = 0$, la distancia pedida es $|X|$.

**Resultado.** Los pies de Enzo, en el píxel $(u, v) = (810, 520)$, se ubican en la
cancha en $(X, Y) \approx (19{,}16;\ 45{,}98)$ m, por lo que patea a aproximadamente
**19,2 m de la línea del arco**.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

In [ ]:
# b) Ubicación de los jugadores argentinos y círculos de 1 m
# ============================================================

# ------------------------------------------------------------
# Píxeles (u, v, 1) de los pies de los jugadores argentinos
# ------------------------------------------------------------
# Cada columna es un jugador, leído a ojo sobre la imagen con la grilla.
# La 5ta columna (810, 520) son los pies de Enzo, el mismo píxel del inciso a).
Pimg = np.array([[100, 225, 380, 380, 810, 850],   # u
                 [545, 400, 325, 300, 520, 350],   # v
                 [  1,   1,   1,   1,   1,   1]], dtype=np.float64)

# ------------------------------------------------------------
# Llevamos los jugadores del plano de la imagen a la cancha
# ------------------------------------------------------------
# (X, Y, 1) ~ H^-1 * (u, v, 1). Usamos la H estimada en el inciso a).
PtsReales = multiplicacionMatricial(inversa(H), Pimg)

# Normalizamos las coordenadas homogéneas dividiendo por la tercera fila
PtsReales_norm = PtsReales / PtsReales[2, :]

# Posición de cada jugador sobre la cancha, en metros
X_reales = PtsReales_norm[0, :]
Y_reales = PtsReales_norm[1, :]

# ------------------------------------------------------------
# Pensamos un círculo de 1 m de radio, definido en coordenadas de la cancha

radio = 1.0
y_circ = np.arange(-100, 101) / 100 * radio
x_circ = np.sqrt(radio**2 - y_circ**2)

X_circ = np.hstack([x_circ, -x_circ[::-1]])
Y_circ = np.hstack([y_circ,  y_circ[::-1]])

# ------------------------------------------------------------
# Proyección de los círculos a la imagen con H
# ------------------------------------------------------------
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)

for i in range(6):
    # Centramos el círculo en la posición real del jugador i: (X, Y, 1)
    CirculoReal_i = np.vstack([X_reales[i] + X_circ,
                               Y_reales[i] + Y_circ,
                               np.ones(X_circ.shape[0])])

    # Proyectamos a la imagen: (u, v, 1) ~ H * (X, Y, 1), y normalizamos
    CirculoImg_i = multiplicacionMatricial(H, CirculoReal_i)
    CirculoImg_i = CirculoImg_i / CirculoImg_i[2, :]

    plt.plot(CirculoImg_i[0, :], CirculoImg_i[1, :], color='black', linewidth=2)

u_jugadores = Pimg[0, :]
v_jugadores = Pimg[1, :]
plt.plot(u_jugadores, v_jugadores, color='black', linestyle='-', linewidth=2, alpha=0.7)

plt.title("Ubicación espacial y círculos tácticos de 1m")
plt.show()

Ubicación de los jugadores argentinos

**Idea.** Ubicamos a cada jugador en la cancha y dibujamos, sobre el piso, un
círculo de 1 m de radio centrado en sus pies. Como todo ocurre sobre el plano de
la cancha, usamos la misma homografía $H$ del inciso a).

**Procedimiento.**
1. Marcamos en la imagen los píxeles $(u, v)$ de los pies de los 6 jugadores argentinos.
2. Los llevamos a la cancha con $H^{-1}$ y normalizamos por la tercera componente,
   obteniendo $(X_i, Y_i)$ en metros.
3. Definimos en coordenadas de la cancha un círculo de radio $r = 1$ m centrado
   en cada jugador: $(X_i + x, \; Y_i + y)$ con $x^2 + y^2 = r^2$.
4. Proyectamos esos puntos a la imagen con $H$ y los dibujamos. Como la
   proyección no conserva las circunferencias, en la imagen se ven como elipses.

c) Trazar una recta paralela a la línea de fondo de la cancha que to que el pie del
último defensor (es decir, el penúltimo jugador adversario, contando al arquero)
para determinar si los jugadores de blanco estaban en fuera de juego. La recta se
define en coordenadas de la cancha y se proyecta a la imagen mediante H.

In [ ]:
# Fuera de juego: recta paralela a la línea de fondo
# ============================================================

img = cv2.imread('futbol/test-fuera-de-juego.png')
img_rgb = img[:, :, ::-1] 

# Imagen con grilla para leer a ojo las coordenadas (u, v)
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.xlabel('u (píxeles horizontal)')
plt.ylabel('v (píxeles vertical)')
plt.title('Identificación de píxeles (u, v)')
plt.show()

# ------------------------------------------------------------

# Origen (0, 0) en la esquina inferior izquierda; el arco de esta foto está en X = 105.
objPts2 = np.array([
    [ 88.5, 54.16],  # P1: esquina superior izq del área grande
    [ 88.5, 13.84],  # P2: esquina inferior izq del área grande
    [ 99.5, 43.16],  # P3: esquina superior izq del área chica
    [ 99.5, 24.84],  # P4: esquina inferior izq del área chica
    [ 94.0, 34.00],  # P5: punto penal
    [105.0, 43.16],  # P6: línea superior del área chica con la línea de meta
], dtype=np.float32)

# Píxeles (u, v) de los mismos puntos, leídos de la grilla (mismo orden que objPts2)
imgPts2 = np.array([
    [ 735, 240],  # P1
    [ 730, 760],  # P2
    [1110, 310],  # P3
    [1180, 520],  # P4
    [ 980, 465],  # P5
    [1385, 185],  # P6
], dtype=np.float32)

# ------------------------------------------------------------
# Homografía de esta foto
# DLT con normalización de Hartley. La llamamos H_fuera para no pisar la H de 4a)
H_fuera = estimarH(imgPts2.astype(np.float64), objPts2.astype(np.float64))

# ------------------------------------------------------------
# Posición del pie del último defensor en la cancha
u_def = 825   # píxel (u, v) del pie del último defensor
v_def = 315

# Llevamos el píxel a la cancha con la inversa de H y deshomogeneizamos
pt_imagen = np.array([[u_def], [v_def], [1]])

pt_real = multiplicacionMatricial(inversa(H_fuera), pt_imagen)

xReal = float(pt_real[0, 0] / pt_real[2, 0])
yReal = float(pt_real[1, 0] / pt_real[2, 0])

print(f"Pie del defensor en la cancha (X, Y): ({xReal:.2f}, {yReal:.2f}) m")
print(f"Distancia a la línea de meta: {105.0 - xReal:.2f} m")


# Paralela a la línea de fondo (X = cte) que pasa por el pie del defensor.
# Recorremos todo el ancho de la cancha, Y de 0 a 68 m (101 puntos).
Y_recta = np.arange(0, 101) / 100 * 68.0
X_recta = xReal * np.ones(101)

# Coordenadas homogéneas (X, Y, 1), matriz de 3 x 101
RectaReal = np.vstack([X_recta, Y_recta, np.ones(101)])

# ------------------------------------------------------------
# Proyección de la recta a la imagen con H
# ------------------------------------------------------------
RectaImg = multiplicacionMatricial(H_fuera, RectaReal)
RectaImg = RectaImg / RectaImg[2, :]   # deshomogeneizamos

u_recta = RectaImg[0, :]
v_recta = RectaImg[1, :]

# Nos quedamos solo con los puntos que caen sobre el pasto en el encuadre
v_carteles_top = 250   # altura aproximada de los carteles LED
v_pasto_bottom = 750   # límite inferior del pasto en el encuadre
mask = (v_recta >= v_carteles_top) * (v_recta <= v_pasto_bottom)   # producto = "y" lógico

u_recta_cancha = u_recta[mask]
v_recta_cancha = v_recta[mask]

# ------------------------------------------------------------
plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.plot(u_recta_cancha, v_recta_cancha, color='red', linewidth=2,
         label=f'Línea de fuera de juego (X = {xReal:.2f} m)')
plt.scatter([u_def], [v_def], color='yellow', s=40, zorder=5)   # pie del defensor
plt.title("Fuera de juego: recta paralela a la línea de fondo")
plt.legend()
plt.show()

Línea de fuera de juego es paralela a la línea de fondo y pasa por el
pie del último defensor (el penúltimo adversario, contando al arquero). En
coordenadas de la cancha es una recta $X = X_{def}$. La proyectamos a la imagen
con la homografía de esta foto.

**Procedimiento.**
1. Identificamos 6 puntos de referencia en la imagen (esquinas del área grande y
   del área chica, punto penal e intersección con la línea de meta) y los
   emparejamos con sus coordenadas en metros según `estadio-azteca.png`.
2. Estimamos $H$ con DLT y normalización de Hartley, y verificamos con el RMS de reproyección.
3. Llevamos el pie del último defensor a la cancha con $H^{-1}$ y normalizamos
   por la tercera componente, obteniendo $X_{def}$.
4. Definimos la recta $\{(X_{def}, Y) : Y \in [0, 68]\}$ en la cancha y la
   proyectamos a la imagen con $H$, recortando la parte que cae sobre el pasto.
5. Comparamos la posición de los jugadores de blanco con la recta.

**Resultado.** El pie del defensor se ubica en $(X, Y) \approx (91{,}2;\ 46{,}6)$ m,
a unos 13,8 m de la línea de meta.

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---